# Venezuela earthquake (2026-06-24) — before/after imagery comparison

Compare the **pre-earthquake Planet quarterly basemap (Q1 2026, ~4.8 m)** against **post-earthquake SkySat/Pelican imagery (50 cm, 26–28 June 2026)** to find evidence of landslides and flag areas for field investigation.

- Data: [source.coop/planet/venezuela-earthquake-2026-06-24](https://source.coop/planet/venezuela-earthquake-2026-06-24) — imagery © Planet Labs PBC, **CC BY-NC 4.0**
- Nothing is downloaded: maps stream tiles straight from the cloud-optimized GeoTIFFs (needs internet).
- Run cells top to bottom. Change `LOCATION` / `SCENE` below and re-run from there to switch areas.

In [1]:
import leafmap

from geer_venezuela import (
    ATTRIBUTION,
    PRE_EVENT_MOSAIC,
    asset_href,
    load_items,
    locations,
    scenes_for,
)

items = load_items("post-event")
locations(items)

,location,location_slug,scenes,first,last,constellations
0,Caracas,caracas,3,2026-06-26 12:35:41.566000+00:00,2026-06-27 14:58:26.402757+00:00,"pelican, skysat"
1,Catia La Mar,catia-la-mar,1,2026-06-26 11:20:57.920000+00:00,2026-06-26 11:20:57.920000+00:00,skysat
2,Independencia & Ocumare de la Costa,independencia-ocumare,1,2026-06-28 12:20:47.540000+00:00,2026-06-28 12:20:47.540000+00:00,skysat
3,La Guaira,la-guaira,8,2026-06-26 15:05:35.183554+00:00,2026-06-27 11:27:22.914000+00:00,"pelican, skysat"
4,Puerto Cabello,puerto-cabello,2,2026-06-26 11:47:59.786000+00:00,2026-06-26 11:47:59.786000+00:00,skysat
5,Valencia,valencia,1,2026-06-26 19:10:38.440000+00:00,2026-06-26 19:10:38.440000+00:00,skysat
6,Yumare,yumare,1,2026-06-28 11:51:15.058000+00:00,2026-06-28 11:51:15.058000+00:00,skysat


## 1. Overview — where the post-event imagery is

Red outlines are post-event scene footprints on top of the pre-event basemap. Click a footprint to see the scene id and acquisition time.

In [2]:
overview = leafmap.Map(center=(10.5, -67.4), zoom=8)
overview.add_cog_layer(
    PRE_EVENT_MOSAIC,
    name="Pre-event basemap (Q1 2026)",
    attribution=ATTRIBUTION,
    zoom_to_layer=False,
)
footprints = items[
    ["id", "title", "location", "datetime", "constellation", "eo:cloud_cover", "geometry"]
].assign(datetime=lambda d: d["datetime"].astype(str))
overview.add_gdf(
    footprints,
    layer_name="Post-event scene footprints",
    style={"color": "#ff3b30", "weight": 2, "fillOpacity": 0.05},
    zoom_to_layer=False,
)
overview

Map(center=[10.5, -67.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## 2. Pick a location and scene

Set `LOCATION` to one of: `caracas`, `catia-la-mar`, `independencia-ocumare`, `la-guaira`, `puerto-cabello`, `valencia`, `yumare`.

In [13]:
LOCATION = "valencia"

scenes = scenes_for(items, LOCATION)
scenes[["id", "datetime", "constellation", "gsd", "eo:cloud_cover", "title"]]

,id,datetime,constellation,gsd,eo:cloud_cover,title
0,20260626_191038_ssc8_u0001,2026-06-26 19:10:38.440000+00:00,skysat,0.72,10,Valencia — 2026-06-26 19:10 UTC


## 3. Before / after swipe

Drag the divider to swipe between **before (left)** and **after (right)**. Set `SCENE` to a row number from the table above (prefer low `eo:cloud_cover`).

In [14]:
SCENE = 0

scene = scenes.iloc[SCENE]
# swipe = leafmap.Map()
# swipe.split_map(
#     left_layer=PRE_EVENT_MOSAIC,
#     right_layer=asset_href(scene, "visual"),
#     left_label="BEFORE — Q1 2026 basemap (4.8 m)",
#     right_label=f"AFTER — {scene['title']} ({scene['constellation']}, 50 cm)",
# )
# swipe

## 4. Flicker comparison (often better for spotting change)

Both layers are stacked. Open the layer control (top right) and **check/uncheck the AFTER layer** to flicker between dates — new landslide scars pop out. The toolbar also has per-layer opacity sliders.

In [15]:
m = leafmap.Map()
m.add_cog_layer(
    PRE_EVENT_MOSAIC,
    name="BEFORE — Q1 2026 basemap",
    attribution=ATTRIBUTION,
    zoom_to_layer=False,
)
m.add_cog_layer(
    asset_href(scene, "visual"),
    name=f"AFTER — {scene['title']}",
    attribution=ATTRIBUTION,
)
m

Map(center=[-68.01176993023023, 10.161973337994594], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## 5. Mark landslide candidates for field teams

On the map above, use the **draw tools** (left edge) to drop points or polygons on suspected landslides, then run the next cell to export them as GeoJSON (loads directly in QGIS, ArcGIS Online, Google Earth, etc.).

In [6]:
from pathlib import Path

out_dir = Path.cwd().parent / "data" / "landslide_candidates"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"{LOCATION}_{scene['id']}.geojson"

if m.user_rois is not None and m.user_rois["features"]:
    m.save_draw_features(str(out_file), indent=2)
    print(f"Saved {len(m.user_rois['features'])} feature(s) to {out_file}")
else:
    print("Nothing drawn yet — use the draw tools on the map above, then re-run this cell.")

Nothing drawn yet — use the draw tools on the map above, then re-run this cell.


## 6. High-resolution BEFORE — Esri Wayback

Google Earth's sharp pre-event imagery is **not open** (Maxar/Airbus licensed to Google — screenshots with attribution only, no GIS use). The legitimate equivalent is **[Esri World Imagery Wayback](https://livingatlas.arcgis.com/wayback/)**: archived releases of the same commercial-grade basemap, served as public tiles. The **2026-05-28 release** is the last full pre-earthquake snapshot and is sub-meter along this coast — a much better "before" than the 4.8 m Planet basemap for confirming small scars.

Caveats: underlying **capture dates vary by tile** (could be months–years old — check the date at the Wayback site before citing a "before" date), and terms allow map display/visual analysis with attribution, not bulk download.

In [ ]:
from geer_venezuela import WAYBACK_ATTRIBUTION, WAYBACK_PRE_EVENT

wb = leafmap.Map()
wb.add_tile_layer(
    WAYBACK_PRE_EVENT,
    name="BEFORE — Esri Wayback 2026-05-28 (sub-meter)",
    attribution=WAYBACK_ATTRIBUTION,
    max_zoom=19,
)
wb.add_cog_layer(
    asset_href(scene, "visual"),
    name=f"AFTER — {scene['title']}",
    attribution=ATTRIBUTION,
)
wb

In [ ]:
wayback_file = out_dir / f"{LOCATION}_{scene['id']}_wayback.geojson"
if wb.user_rois is not None and wb.user_rois["features"]:
    wb.save_draw_features(str(wayback_file), indent=2)
    print(f"Saved {len(wb.user_rois['features'])} feature(s) to {wayback_file}")
else:
    print("Nothing drawn on the Wayback map yet — draw, then re-run this cell.")

## What to look for

- **Fresh scars**: light-toned (tan/grey) patches of bare soil/rock on vegetated slopes that are absent in the BEFORE image.
- **Debris runouts**: fans or flow paths below scars — down drainages, across roads, into buildings.
- **Blocked drainages / turbid water**: sediment plumes at river mouths, ponding upstream of debris.
- **Road cuts and coastal bluffs**: common failure points; scan along the Caracas–La Guaira corridor.

Caveats:

- The BEFORE image is **4.8 m** vs 50 cm AFTER — small slides are only visible in the AFTER image; use the BEFORE mainly to confirm a scar is *new*.
- Check `eo:cloud_cover` and prefer clear scenes; the `udm2` asset of each scene is a per-pixel cloud/shadow mask if needed.
- The Q1 2026 basemap is a quarterly composite (Jan–Mar), so seasonal vegetation differences are possible.

## Ideas for additional data (later)

- **Sentinel-2** (10 m, free, ~5-day revisit) via Earth Search STAC — regional sweep beyond the Planet footprints.
- **Maxar Open Data Program** — often releases 30–50 cm imagery for major disasters.
- **Copernicus EMS / UNOSAT** rapid-mapping activations — may already have damage/landslide vectors.
- **NASA/USGS**: ShakeMap + slope data to prioritize where landslides are *likely*, not just visible.